Dataset choice and pre-processing data 

In [ ]:
import pandas as pd
# Import the dataset
df = pd.read_csv("online_shoppers_intention.csv")

print("Column names:", df.columns.tolist())
print("Dataset shape:", df.shape)

# Data preprocessing
# One-hot encode categorical variables
categorical_cols = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

# Keep features and target separate after encoding
X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

print("Shape after one-hot encoding:", df.shape)

# Split the dataset into training and testing sets
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set: {X_test.shape[0]} rows")

Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression as LRClassifier
from sklearn.pipeline import make_pipeline as build_lr_pipeline
from sklearn.preprocessing import StandardScaler as LRScaler
from sklearn.metrics import (
    accuracy_score as lr_accuracy_metric,
    roc_auc_score as lr_auc_metric,
    precision_score as lr_precision_metric,
    recall_score as lr_recall_metric,
    f1_score as lr_f1_metric,
    matthews_corrcoef as lr_mcc_metric,
)

# Copy the data for Logistic Regression
lr_training_matrix = X_train.copy()
lr_testing_matrix = X_test.copy()
lr_training_labels = y_train.astype(int).copy()
lr_testing_labels = y_test.astype(int).copy()

# Build and train a scaled Logistic Regression pipeline
lr_fitted_pipeline = build_lr_pipeline(
    LRScaler(),
    LRClassifier(max_iter=1000, random_state=42),
)
lr_fitted_pipeline.fit(lr_training_matrix, lr_training_labels)

# Generate test predictions and positive-class probabilities
lr_test_predictions = lr_fitted_pipeline.predict(lr_testing_matrix)
lr_test_probabilities = lr_fitted_pipeline.predict_proba(lr_testing_matrix)[:, 1]

# Evaluate the trained model
lr_metric_results = {
    "Accuracy": lr_accuracy_metric(lr_testing_labels, lr_test_predictions),
    "AUC Score": lr_auc_metric(lr_testing_labels, lr_test_probabilities),
    "Precision": lr_precision_metric(
        lr_testing_labels, lr_test_predictions, zero_division=0
    ),
    "Recall": lr_recall_metric(
        lr_testing_labels, lr_test_predictions, zero_division=0
    ),
    "F1 Score": lr_f1_metric(
        lr_testing_labels, lr_test_predictions, zero_division=0
    ),
    "Matthews Correlation Coefficient": lr_mcc_metric(
        lr_testing_labels, lr_test_predictions
    ),
}

for lr_metric_name, lr_metric_value in lr_metric_results.items():
    print(f"{lr_metric_name}: {lr_metric_value:.4f}")

In [ ]:
import joblib

# Save the fitted pipeline for later use
lr_model_path = "logistic_regression_metrics_pipeline.joblib"
joblib.dump(lr_fitted_pipeline, lr_model_path)
print(f"Saved trained model to: {lr_model_path}")

In [ ]:
# Load the saved pipeline and use it to predict
lr_loaded_pipeline = joblib.load("logistic_regression_metrics_pipeline.joblib")
lr_loaded_predictions = lr_loaded_pipeline.predict(lr_testing_matrix)
print(f"Loaded model produced {len(lr_loaded_predictions)} predictions.")

Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier as DecisionTreeEstimator
from sklearn.metrics import (
    accuracy_score as tree_accuracy,
    roc_auc_score as tree_auc,
    precision_score as tree_precision,
    recall_score as tree_recall,
    f1_score as tree_f1,
    matthews_corrcoef as tree_mcc,
)

# Prepare the training data and fit the decision tree classifier.
tree_training_features = X_train.copy()
tree_training_labels = y_train.astype(int).copy()

tree_classifier = DecisionTreeEstimator(random_state=42)
tree_classifier.fit(tree_training_features, tree_training_labels)

# Generate predictions on the held-out test data and evaluate the classifier.
tree_testing_features = X_test.copy()
tree_testing_labels = y_test.astype(int).copy()

tree_predicted_labels = tree_classifier.predict(tree_testing_features)
tree_positive_class_index = list(tree_classifier.classes_).index(1)
tree_positive_probabilities = tree_classifier.predict_proba(
    tree_testing_features
)[:, tree_positive_class_index]

tree_evaluation = {
    "Accuracy": tree_accuracy(tree_testing_labels, tree_predicted_labels),
    "AUC Score": tree_auc(tree_testing_labels, tree_positive_probabilities),
    "Precision": tree_precision(
        tree_testing_labels, tree_predicted_labels, zero_division=0
    ),
    "Recall": tree_recall(
        tree_testing_labels, tree_predicted_labels, zero_division=0
    ),
    "F1 Score": tree_f1(
        tree_testing_labels, tree_predicted_labels, zero_division=0
    ),
    "Matthews Correlation Coefficient": tree_mcc(
        tree_testing_labels, tree_predicted_labels
    ),
}

for tree_metric, tree_value in tree_evaluation.items():
    print(f"{tree_metric}: {tree_value:.4f}")

In [ ]:
import joblib

# Persist the fitted classifier so it can be loaded without retraining.
tree_model_file = "decision_tree_classifier.joblib"
joblib.dump(tree_classifier, tree_model_file)
print(f"Saved trained model to: {tree_model_file}")

In [ ]:
# Load the saved classifier and verify it can produce predictions.
loaded_tree_classifier = joblib.load(tree_model_file)
loaded_tree_predictions = loaded_tree_classifier.predict(tree_testing_features)
print(f"Loaded model produced {len(loaded_tree_predictions)} predictions.")

K-Nearest Neighbor Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier as NeighborVoteModel
from sklearn.pipeline import make_pipeline as assemble_knn_pipeline
from sklearn.preprocessing import StandardScaler as DistanceFeatureScaler
from sklearn.metrics import (
    accuracy_score as knn_accuracy_metric,
    roc_auc_score as knn_auc_metric,
    precision_score as knn_precision_metric,
    recall_score as knn_recall_metric,
    f1_score as knn_f1_metric,
    matthews_corrcoef as knn_mcc_metric,
)

# Prepare distinct training inputs and fit a scaled K-Nearest Neighbor model.
knn_fit_features = X_train.copy()
knn_fit_labels = y_train.astype(int).copy()
knn_trained_pipeline = assemble_knn_pipeline(
    DistanceFeatureScaler(),
    NeighborVoteModel(n_neighbors=5),
)

knn_trained_pipeline.fit(knn_fit_features, knn_fit_labels)

# Test the trained model on the held-out data and calculate all six metrics.
knn_holdout_features = X_test.copy()
knn_holdout_labels = y_test.astype(int).copy()
knn_holdout_predictions = knn_trained_pipeline.predict(knn_holdout_features)
knn_positive_label_position = list(knn_trained_pipeline.classes_).index(1)
knn_positive_class_scores = knn_trained_pipeline.predict_proba(
    knn_holdout_features
)[:, knn_positive_label_position]

knn_scorecard = {
    "Accuracy": knn_accuracy_metric(knn_holdout_labels, knn_holdout_predictions),
    "AUC Score": knn_auc_metric(knn_holdout_labels, knn_positive_class_scores),
    "Precision": knn_precision_metric(
        knn_holdout_labels, knn_holdout_predictions, zero_division=0
    ),
    "Recall": knn_recall_metric(
        knn_holdout_labels, knn_holdout_predictions, zero_division=0
    ),
    "F1 Score": knn_f1_metric(
        knn_holdout_labels, knn_holdout_predictions, zero_division=0
    ),
    "Matthews Correlation Coefficient": knn_mcc_metric(
        knn_holdout_labels, knn_holdout_predictions
    ),
}

for knn_score_name, knn_score_value in knn_scorecard.items():
    print(f"{knn_score_name}: {knn_score_value:.4f}")

In [ ]:
import joblib

# Save the complete pipeline, including its feature scaler, for reuse.
knn_saved_model_path = "knn_revenue_classifier.joblib"
joblib.dump(knn_trained_pipeline, knn_saved_model_path)
print(f"Saved trained model to: {knn_saved_model_path}")

In [ ]:
# Load the persisted pipeline and confirm it predicts without retraining.
knn_restored_pipeline = joblib.load(knn_saved_model_path)
knn_restored_predictions = knn_restored_pipeline.predict(knn_holdout_features)
print(f"Loaded model produced {len(knn_restored_predictions)} predictions.")

Naive Bayes Classifier

In [ ]:
from sklearn.naive_bayes import GaussianNB as RevenueNaiveBayes
from sklearn.metrics import (
    accuracy_score as nb_accuracy,
    roc_auc_score as nb_auc,
    precision_score as nb_precision,
    recall_score as nb_recall,
    f1_score as nb_f1,
    matthews_corrcoef as nb_mcc,
)

# Prepare the training data and fit the Naive Bayes classifier.
nb_training_features = X_train.copy()
nb_training_labels = y_train.astype(int).copy()

nb_trained_classifier = RevenueNaiveBayes()
nb_trained_classifier.fit(nb_training_features, nb_training_labels)

# Test the trained model on held-out data and calculate all six metrics.
nb_testing_features = X_test.copy()
nb_testing_labels = y_test.astype(int).copy()

nb_predicted_labels = nb_trained_classifier.predict(nb_testing_features)
nb_positive_class_index = list(nb_trained_classifier.classes_).index(1)
nb_positive_class_scores = nb_trained_classifier.predict_proba(
    nb_testing_features
)[:, nb_positive_class_index]

nb_metric_results = {
    "Accuracy": nb_accuracy(nb_testing_labels, nb_predicted_labels),
    "AUC Score": nb_auc(nb_testing_labels, nb_positive_class_scores),
    "Precision": nb_precision(
        nb_testing_labels, nb_predicted_labels, zero_division=0
    ),
    "Recall": nb_recall(
        nb_testing_labels, nb_predicted_labels, zero_division=0
    ),
    "F1 Score": nb_f1(
        nb_testing_labels, nb_predicted_labels, zero_division=0
    ),
    "Matthews Correlation Coefficient": nb_mcc(
        nb_testing_labels, nb_predicted_labels
    ),
}

for nb_metric_name, nb_metric_value in nb_metric_results.items():
    print(f"{nb_metric_name}: {nb_metric_value:.4f}")

In [ ]:
import joblib

# Save the fitted classifier so it can be loaded without retraining.
nb_model_path = "naive_bayes_classifier.joblib"
joblib.dump(nb_trained_classifier, nb_model_path)
print(f"Saved trained model to: {nb_model_path}")

In [ ]:
# Load the saved classifier and confirm it can predict.
nb_loaded_classifier = joblib.load(nb_model_path)
nb_loaded_predictions = nb_loaded_classifier.predict(nb_testing_features)
print(f"Loaded model produced {len(nb_loaded_predictions)} predictions.")

Support Vector Machine

In [ ]:
from sklearn.pipeline import make_pipeline as create_svm_pipeline
from sklearn.preprocessing import StandardScaler as SVMFeatureScaler
from sklearn.svm import SVC as RevenueSVMClassifier
from sklearn.metrics import (
    accuracy_score as svm_accuracy_metric,
    roc_auc_score as svm_auc_metric,
    precision_score as svm_precision_metric,
    recall_score as svm_recall_metric,
    f1_score as svm_f1_metric,
    matthews_corrcoef as svm_mcc_metric,
)

# Prepare training data and fit a scaled Support Vector Machine.
svm_training_features = X_train.copy()
svm_training_labels = y_train.astype(int).copy()

svm_fitted_pipeline = create_svm_pipeline(
    SVMFeatureScaler(),
    RevenueSVMClassifier(probability=True, random_state=42),
)
svm_fitted_pipeline.fit(svm_training_features, svm_training_labels)

print("Support Vector Machine training complete.")

# Generate predictions and positive-class probabilities for held-out data.
svm_testing_features = X_test.copy()
svm_testing_labels = y_test.astype(int).copy()

svm_predicted_labels = svm_fitted_pipeline.predict(svm_testing_features)
svm_positive_class_index = list(svm_fitted_pipeline.classes_).index(1)
svm_positive_class_probabilities = svm_fitted_pipeline.predict_proba(
    svm_testing_features
)[:, svm_positive_class_index]

# Evaluate the trained model on the test data.
svm_metric_results = {
    "Accuracy": svm_accuracy_metric(svm_testing_labels, svm_predicted_labels),
    "AUC Score": svm_auc_metric(
        svm_testing_labels, svm_positive_class_probabilities
    ),
    "Precision": svm_precision_metric(
        svm_testing_labels, svm_predicted_labels, zero_division=0
    ),
    "Recall": svm_recall_metric(
        svm_testing_labels, svm_predicted_labels, zero_division=0
    ),
    "F1 Score": svm_f1_metric(
        svm_testing_labels, svm_predicted_labels, zero_division=0
    ),
    "Matthews Correlation Coefficient": svm_mcc_metric(
        svm_testing_labels, svm_predicted_labels
    ),
}

for svm_metric_name, svm_metric_value in svm_metric_results.items():
    print(f"{svm_metric_name}: {svm_metric_value:.4f}")

In [ ]:
import joblib

# Save the full pipeline, including its scaler, for later use.
svm_model_path = "svm_revenue_classifier.joblib"
joblib.dump(svm_fitted_pipeline, svm_model_path)
print(f"Saved trained model to: {svm_model_path}")

In [ ]:
# Load the saved pipeline and confirm it can predict on the test features.
svm_loaded_pipeline = joblib.load(svm_model_path)
svm_loaded_predictions = svm_loaded_pipeline.predict(svm_testing_features)
print(f"Loaded model produced {len(svm_loaded_predictions)} predictions.")